Bag of words y TF-IDF

In [2]:
import json
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

archivos = {
    "espont":    "../data/processed/fallacyes_espont_procesado.jsonl",
    "political": "../data/processed/fallacyes_political_procesado.jsonl",
    "proto":     "../data/processed/fallacyes_proto_procesado.jsonl",
}

def juntar(registro, campo):
    return [palabra for frase in registro["oraciones"] for palabra in frase[campo]]

datos = {}
matrices = {}

for nombre, ruta in archivos.items():
    with open(ruta, 'r', encoding='utf-8') as f:
        registros = [json.loads(linea) for linea in f]

    raices = [juntar(r, "raices_bow") for r in registros]
    tokens = [juntar(r, "tokens") for r in registros]

    #Coger los datos del train
    raices_train = [raiz for r, raiz in zip(registros, raices) if r["split"] == "train"]

    vect_bow = CountVectorizer(analyzer=lambda x: x).fit(raices_train)
    vect_tfidf = TfidfVectorizer(analyzer=lambda x: x).fit(raices_train)

    datos[nombre] = {"registros": registros, "raices": raices, "tokens": tokens}
    matrices[nombre] = {
        "vect_bow": vect_bow, "bow": vect_bow.transform(raices),
        "vect_tfidf": vect_tfidf, "tfidf": vect_tfidf.transform(raices),
    }

    print(f"[{nombre}] textos: {len(registros)}, de entrenamiento: {len(raices_train)}")
    print(f"-Raíces del vocabulario (entrenamiento): {len(vect_bow.get_feature_names_out())}")
    print(f"-Dimensiones de la tabla: {matrices[nombre]['bow'].shape}")

[espont] textos: 1840, de entrenamiento: 1301
-Raíces del vocabulario (entrenamiento): 4200
-Dimensiones de la tabla: (1840, 4200)
[political] textos: 1951, de entrenamiento: 1316
-Raíces del vocabulario (entrenamiento): 3371
-Dimensiones de la tabla: (1951, 3371)
[proto] textos: 3319, de entrenamiento: 2341
-Raíces del vocabulario (entrenamiento): 3935
-Dimensiones de la tabla: (3319, 3935)


Palabras que no aparecen

In [3]:
import os
from gensim.models import KeyedVectors
from huggingface_hub import hf_hub_download

ruta_kv = "../models/eswiki_300d.kv"
ruta_txt = "../models/eswiki_20180420_300d.txt"

if os.path.exists(ruta_kv):
    print("Cargando modelo rápido...")
    modelo_w2v = KeyedVectors.load(ruta_kv)
else:
    if os.path.exists(ruta_txt):
        print("Cargando desde el archivo de texto (lento)")
        ruta = ruta_txt
    else:
        print("Descargando desde Hugging Face")
        ruta = hf_hub_download(
            repo_id="Word2vec/wikipedia2vec_eswiki_20180420_300d",
            filename="eswiki_20180420_300d.txt"
        )
    modelo_w2v = KeyedVectors.load_word2vec_format(ruta)
    os.makedirs("../models", exist_ok=True)
    modelo_w2v.save(ruta_kv)

print("Modelo cargado")

Cargando modelo rápido...
Modelo cargado


In [2]:
modelo_w2v.save("../models/eswiki_300d.kv")

Palabras no representadas

In [4]:
def tiene_vector(p):
    return p in modelo_w2v.key_to_index or p.lower() in modelo_w2v.key_to_index

for nombre, contenido in datos.items():
    palabras = [p for frase in contenido["tokens"] for p in frase if p.isalpha()]
    vocabulario_unico = set(palabras)

    oov = [p for p in vocabulario_unico if not tiene_vector(p)]
    cubiertas = sum(tiene_vector(p) for p in palabras)

    print(f"--- {nombre.upper()} ---")
    print(f"Palabras fuera del vocabulario (OOV): {len(oov)}")
    print(f"Porcentaje de pérdida: {len(oov)/len(vocabulario_unico)*100:.2f}%")
    print(f"Cobertura real de tokens: {cubiertas/len(palabras)*100:.2f}%\n")

--- ESPONT ---
Palabras fuera del vocabulario (OOV): 405
Porcentaje de pérdida: 4.23%
Cobertura real de tokens: 99.12%

--- POLITICAL ---
Palabras fuera del vocabulario (OOV): 232
Porcentaje de pérdida: 2.82%
Cobertura real de tokens: 99.66%

--- PROTO ---
Palabras fuera del vocabulario (OOV): 192
Porcentaje de pérdida: 2.07%
Cobertura real de tokens: 99.65%



Un vector por texto

In [5]:
import numpy as np

def buscar(palabra):
    if palabra in modelo_w2v.key_to_index:
        return palabra
    if palabra.lower() in modelo_w2v.key_to_index:
        return palabra.lower()
    return None

def vector_del_texto(lista_tokens):
    claves = [buscar(p) for p in lista_tokens]
    claves = [c for c in claves if c is not None]
    if not claves:
        return np.zeros(modelo_w2v.vector_size)
    return np.mean([modelo_w2v[c] for c in claves], axis=0)

for nombre, d in datos.items():
    matrices[nombre]["w2v"] = np.array([vector_del_texto(t) for t in d["tokens"]])
    m = matrices[nombre]["w2v"]
    print(f"[{nombre}] dimensiones: {m.shape}, textos sin vector: {int((m == 0).all(axis=1).sum())}")

[espont] dimensiones: (1840, 300), textos sin vector: 0
[political] dimensiones: (1951, 300), textos sin vector: 0
[proto] dimensiones: (3319, 300), textos sin vector: 0


Vector conjunto

In [6]:
import os
import json
import numpy as np
from scipy import sparse

todas_raices, indice = [], []
for nombre, d in datos.items():
    for r, raiz in zip(d["registros"], d["raices"]):
        todas_raices.append(raiz)
        indice.append({"id": r["id"], "conjunto": nombre, "split": r["split"], "etiquetas": r["etiquetas"]})

raices_train = [raiz for raiz, fila in zip(todas_raices, indice) if fila["split"] == "train"]

vect_bow_conjunto = CountVectorizer(analyzer=lambda x: x).fit(raices_train)
vect_tfidf_conjunto = TfidfVectorizer(analyzer=lambda x: x).fit(raices_train)
bow_conjunto = vect_bow_conjunto.transform(todas_raices)
tfidf_conjunto = vect_tfidf_conjunto.transform(todas_raices)
w2v_conjunto = np.vstack([matrices[n]["w2v"] for n in datos])

carpeta = "../data/representations/conjunto"
os.makedirs(carpeta, exist_ok=True)
sparse.save_npz(f"{carpeta}/bow.npz", bow_conjunto)
sparse.save_npz(f"{carpeta}/tfidf.npz", tfidf_conjunto)
np.save(f"{carpeta}/w2v_300d.npy", w2v_conjunto.astype(np.float32))
with open(f"{carpeta}/vocabulario_raices.json", "w", encoding="utf-8") as f:
    json.dump(vect_bow_conjunto.get_feature_names_out().tolist(), f, ensure_ascii=False)
with open(f"{carpeta}/indice.jsonl", "w", encoding="utf-8") as f:
    for fila in indice:
        f.write(json.dumps(fila, ensure_ascii=False) + "\n")

print("Comentarios:", len(indice), "| entrenamiento:", len(raices_train))
print("Bolsa de palabras conjunta:", bow_conjunto.shape)
print("Vectores de palabras conjuntos:", w2v_conjunto.shape)

Comentarios: 7110 | entrenamiento: 4958
Bolsa de palabras conjunta: (7110, 7215)
Vectores de palabras conjuntos: (7110, 300)


Exportación

In [7]:
import os
import json
import numpy as np
from scipy import sparse

for nombre, d in datos.items():
    m = matrices[nombre]
    carpeta = f"../data/representations/{nombre}"
    os.makedirs(carpeta, exist_ok=True)

    sparse.save_npz(f"{carpeta}/bow.npz", m["bow"])
    sparse.save_npz(f"{carpeta}/tfidf.npz", m["tfidf"])
    np.save(f"{carpeta}/w2v_300d.npy", m["w2v"].astype(np.float32))
    with open(f"{carpeta}/vocabulario_raices.json", "w", encoding="utf-8") as f:
        json.dump(m["vect_bow"].get_feature_names_out().tolist(), f, ensure_ascii=False)

    assert len(d["registros"]) == m["w2v"].shape[0]
    ruta_salida = f"../data/processed/fallacyes_{nombre}_vectorizado.jsonl"
    with open(ruta_salida, "w", encoding="utf-8") as f_out:
        for registro, vector in zip(d["registros"], m["w2v"]):
            registro["vector_w2v"] = [round(float(x), 5) for x in vector]
            f_out.write(json.dumps(registro, ensure_ascii=False) + "\n")

    print(f"[{nombre}] guardado: {ruta_salida} y {carpeta}/")

[espont] guardado: ../data/processed/fallacyes_espont_vectorizado.jsonl y ../data/representations/espont/
[political] guardado: ../data/processed/fallacyes_political_vectorizado.jsonl y ../data/representations/political/
[proto] guardado: ../data/processed/fallacyes_proto_vectorizado.jsonl y ../data/representations/proto/


Visualización de datos

In [8]:
import json
from scipy import sparse

nombre = "espont"
carpeta = f"../data/representations/{nombre}"

with open(f'../data/processed/fallacyes_{nombre}_vectorizado.jsonl', encoding='utf-8') as f:
    primera = json.loads(f.readline())
print("Texto:", primera['texto'])
print("Etiquetas:", primera['etiquetas'])
print("Números del vector:", len(primera['vector_w2v']), "->", primera['vector_w2v'][:5])

with open(f"{carpeta}/vocabulario_raices.json", encoding="utf-8") as f:
    vocabulario = json.load(f)
print("\nRaíces:", len(vocabulario), "->", vocabulario[:8], "...", vocabulario[-3:])

for tecnica in ["bow", "tfidf"]:
    matriz = sparse.load_npz(f"{carpeta}/{tecnica}.npz")
    fila = matriz[0]
    print(f"\n{tecnica} - fila 0:")
    for columna, valor in zip(fila.indices, fila.data):
        print("  ", vocabulario[columna], "=", round(float(valor), 3))

Texto: Para mi lo que diga este tipejo tiene cero validez.
Etiquetas: ['ad_hominem']
Números del vector: 300 -> [0.03573, -0.09328, -0.02979, 0.25083, -0.21277]

Raíces: 4200 -> ['!', '1º', '200mil', '24h', '2º', '3º', '5m3', '8km'] ... ['zoo', 'zorr', 'łukasz']

bow - fila 0:
   cer = 1.0
   dig = 1.0
   tipej = 1.0
   validez = 1.0

tfidf - fila 0:
   cer = 0.5
   dig = 0.378
   tipej = 0.551
   validez = 0.551


In [9]:
from collections import Counter
faltan = Counter(p for d in datos.values() for frase in d["tokens"] for p in frase if p.isalpha() and not tiene_vector(p))
print(faltan.most_common(20))

[('COVID', 10), ('Covid', 7), ('batasunos', 5), ('sanchismo', 5), ('basándote', 4), ('copagos', 4), ('autopercibe', 4), ('Estornudé', 4), ('evolucionamos', 4), ('Jaquon', 4), ('covid', 3), ('meneante', 3), ('quejarán', 3), ('gobernamos', 3), ('pactaremos', 3), ('Arhual', 3), ('proetarras', 3), ('desprotegiendo', 3), ('apuntaré', 3), ('suspenderás', 3)]


In [10]:
from collections import Counter
for nombre, d in datos.items():
    print(nombre, Counter(r["split"] for r in d["registros"]))

espont Counter({'train': 1301, 'dev': 273, 'test': 266})
political Counter({'train': 1316, 'test': 344, 'dev': 291})
proto Counter({'train': 2341, 'test': 489, 'dev': 489})
